# 06 - Exporting Specifications to R / Apollo

Delphos is designed to act as an assistant. Once it helps you identify a promising specification, you will likely want to transfer that model back into R Studio (or your preferred R environment) to fine-tune it, generate willingness-to-pay (WTP) estimates, or calculate market shares using Apollo's native functions.

In this notebook, we'll see how to extract the exact `.R` scripts generated by Delphos.

In [ ]:
import delphos as dp
from pathlib import Path

dataset = dp.load_dataset("dataset_4")
agent = dp.load_agent()

models = agent.propose(
    dataset, 
    n_models=1, 
    estimate=False  # We do not need to estimate it to get the R code!
)

proposal = models.proposals[0]


## Generating the Apollo R Script

Delphos provides a utility function `generate_apollo_r_script` that builds the complete R execution code from a proposal's `apollo_specification`.

In [ ]:
from delphos.env.apollo.estimator import generate_apollo_r_script

r_script_content = generate_apollo_r_script(
    task=dataset,
    apollo_specification=proposal.apollo_specification,
    output_directory=Path(".").resolve(),
    summary_file=Path("summary.csv").resolve(),
    save=True
)

# Save it to a file
Path("my_apollo_model.R").write_text(r_script_content)

print(r_script_content[:500] + "\n...\n[Code truncated for display]")


## Running the script in R

You can now simply open `my_apollo_model.R` in RStudio. 

It includes everything you need:
- `apollo_initialise()`
- Data loading (`read.csv` pointing to the exact CSV location)
- The full list of `apollo_beta` and `apollo_fixed` parameters.
- The generated `apollo_probabilities` utility functions.
- The `apollo_estimate()` call.

Once loaded in R, you can add commands like `apollo_modelOutput(model)` to get standard errors, t-tests, and correlation matrices, bridging the gap between Delphos's automated search and manual choice modelling.